# CS 315 Tutorial - Explorations with spaCy (an NLP library)

**Author:** Eni Mustafaraj.  
**Date:** Nov 10, 2026

**Table of Content**
1. [Load and explore the MDL dataset](#sec1)
2. [Explore the Netflix dataset](#sec2)
3. [Finding NERs in text with `spaCy`](#sec3)
4. [Semantic Similarity with `spaCy`](#sec4)
5. [Similarity with Sentence Embeddings](#sec5)


Thi tutorial has two parts:

* Part 1: Explore the MDL dataset to refresh pandas and visualization tools
* Part 2: Introduce the `spaCy` library for NLP tasks such as NER and semantic similarity

<a id="sec1"></a>
## 1. Load and explore the MDL dataset

We will work with the MDL dataset of about 3700 Kdramas, which EM scraped in Summer 2026, so it doesn't have the most recent shows.

In [ ]:
import pandas as pd
df = pd.read_csv("all_dramas.csv")
df.head()

Let's check the names of columns and corresponding values for the first row, which we are displaying as "transposed", to see all the columns.

In [ ]:
df.iloc[[0]].T

We cannot easily see what the values for the rows are, so we will use a pandas function, `set_option` that expands the width of columns to let us see all text:

In [ ]:
pd.set_option('display.max_colwidth', None)
df.iloc[[0]].T

For my purposes of analysis, I don't want all the columns, thus, I'll print out the column names and then select only the ones I want to work with to create a new dataframe:

In [ ]:
df.columns

In [ ]:
columns = ['Year', 'Title_English', 'Original Network', 'Popularity', 'Ranked',
       'Score', 'Score_Votes', 'Watchers', 'Title_Korean', 'Genres',
       'Episodes', 'Tags', 'Related Content', 'Also Known As', 'Screenwriter',
       'Director', 'Screenwriter & Director', 'Aired', 'Airs', 'Aired On',
       'Duration', 'Content Rating', 'Summary', 'source_file']

desiredCols = ['Year', 'Title_English', 'Original Network', 'Genres',
       'Episodes', 'Tags', 'Aired', 'Summary']

Now that I have a smaller subset of columns, I can create a new dataset, that has only these desired columns.
I have to `copy` the new dataframe into a new variable, to not run into warning issues about changing the underlying dataframe.

In [ ]:
dfS = df[desiredCols].copy()
dfS.head()

We can explore the various columns, for example, let's see the range of episode numbers:

In [ ]:
dfS['Episodes'].describe()

And we can count how many shows have only 16 episodes:

In [ ]:
(dfS['Episodes']==16).sum()

Or, we can look at the original networks:

In [ ]:
dfS['Original Network'].value_counts()

It looks like there are a lot of unique entries, because of combinations of multiple networks. 
Let's focus on the top 10 networks:

In [ ]:
dfS['Original Network'].value_counts()[:10]

Let's find the percentage of shows that are produced by the Top 10 networks:

In [ ]:
# Find the total sum of values for the top 10 entries

dfS['Original Network'].value_counts()[:10].sum() 

In [ ]:
# find the percentage of shows produced by the Top 10 networks

dfS['Original Network'].value_counts()[:10].sum() / dfS['Original Network'].value_counts().sum() * 100

Let's create a subset containing only the Netflix Originals:

In [ ]:
netflix = dfS[dfS['Original Network'].str.contains('Netflix', na=False)].copy()
netflix.shape

In [ ]:
netflix.head()

Now that we have a more focused dataset, let's explore it.

<a id="sec2"></a>
## 2. Explore the Netflix dataset

The first thing I want to do is to find out the most common genres in Netflix shows. Because the values stored in the column `Genres` look like a list, we can use the `explode` method to "explode" the individual lists and then count the unique genre values:

In [ ]:
netflix['Genres'].explode().nunique()

This number looks too big for 88 rows, so let's check if the values stored in the column are really lists:

In [ ]:
type(netflix['Genres'].iloc[0])

The values seem to be lists, this is to be expected because we read the original data from a CSV, not a JSON. Let's see if the data can be turned into a list using JSON:

In [ ]:
netflix['Genres'].iloc[0]

Because this string contains single quoted strings, we cannot use JSON functions to convert it, let's see why:

In [ ]:
import json

try:
    str = json.loads(dfS['Genres'].iloc[0])
    print(str)
except json.JSONDecodeError:
    print("JSONDecodeError")

try:
    str = '["blue", "red"]'
    lst = json.loads(str)
    print("Converted string", lst)

except json.JSONDecodeError:
    print("JSONDecodeError")

Since our string is not in a format that we can decode, we'll use a different method. Note: this is where I googled how to convert a string like this into a list and Google's AI gave me this code snippet:

```
import ast

s = "['abc', 'def']"
my_list = ast.literal_eval(s)

print(my_list)         # Output: ['abc', 'def']
print(type(my_list))   # Output: <class 'list'>
```

Thus, there is a function from a new library, `ast`, which stands for **abstract syntax tree**.

I will use the method `apply` to change the entire column to be a list:

In [ ]:
import ast
netflix['Genres'] = netflix['Genres'].apply(ast.literal_eval)
netflix['Genres'].iloc[0]

This conversion worked, so we can go back and count the unique genres:

In [ ]:
netflix['Genres'].explode().nunique()

This result is more credible for a list of 88 shows. Now let's see these genres:

In [ ]:
netflix['Genres'].explode().value_counts()

Let's look at various visualizations techniques for the "Genres" column. The simplest one is a bar chart, but it will only show the genres individually:

### Horizontal barchart

In [ ]:
import matplotlib.pyplot as plt

# Get counts for each value as we did before
counts = netflix['Genres'].explode().value_counts()

# Separate top 12 genres and aggregate the rest, because there are too many
top_genres = counts.head(12)
other_count = counts.iloc[12:].sum()
top_genres['Other (< 6 titles)'] = other_count

# Plot horizontal bars 
plt.figure(figsize=(8, 5))
top_genres.sort_values().plot(kind='barh', color='salmon')
plt.xlabel("Total Count Across All Dramas")
plt.title(f"Netflix K-Drama Genres Distribution (n={netflix.shape[0]})")
plt.tight_layout()
plt.show()

## Other plots

I asked Gemini to suggest other kinds of plots for this data and here are what it suggested:

1. A co-occurrence heatmap
2. A network graph
3. An UpSet plot [see Wikipedia page for this plot](https://en.wikipedia.org/wiki/UpSet_plot)

### Co-occurrence heatmap

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# 1. One-hot encode genres per drama (0 or 1)
genre_dummies = pd.get_dummies(netflix['Genres'].explode()).groupby(level=0).sum()

# 2. Compute co-occurrence matrix via matrix dot product
co_matrix = genre_dummies.T.dot(genre_dummies)

# 3. Plot lower triangle heatmap to remove duplicate pairs
plt.figure(figsize=(12, 10))
sns.heatmap(co_matrix, annot=True, fmt='d', cmap='YlGnBu', cbar=False)
plt.title("Genre Co-Occurrence Matrix")
plt.tight_layout()
plt.show()

### Network graph plot

This requires that you install the `networkx` library.
**Note:** Uses the `co_matrix` defined for the prior graph.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Build graph from the co-occurrence matrix
G = nx.from_pandas_adjacency(co_matrix)

# Remove self-loops (genre co-occurring with itself)
G.remove_edges_from(nx.selfloop_edges(G))

# Filter out weak co-occurrences (e.g., paired only 1 time) for a cleaner layout
threshold = 2
edges_to_keep = [(u, v) for u, v, d in G.edges(data=True) if d['weight'] >= threshold]
G_filtered = G.edge_subgraph(edges_to_keep)

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G_filtered, k=0.5, seed=42)
weights = [G_filtered[u][v]['weight'] * 0.8 for u, v in G_filtered.edges()]

nx.draw_networkx_nodes(G_filtered, pos, node_size=1200, node_color='skyblue')
nx.draw_networkx_labels(G_filtered, pos, font_size=9, font_weight='bold')
nx.draw_networkx_edges(G_filtered, pos, width=weights, edge_color='gray', alpha=0.7)

plt.title("Netflix K-Drama Genre Relationship Network")
plt.axis('off')
plt.show()

This is not very helpful, because it doesn't scale the circles by genre frequency, so I asked Gemini to do that and make the graph better.  
**Note:** It took about 4 iterations to create the graph below, everytime I will get a graph, I'll aks Gemini to change something to make it better. 

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# 1. Explode genres and compute counts
exploded = netflix['Genres'].explode()
genre_counts = exploded.value_counts()

# 2. Build Genre x Drama cross-tabulation and co-occurrence matrix
indicator_matrix = pd.crosstab(exploded.index, exploded)
co_matrix = indicator_matrix.T.dot(indicator_matrix)

# 3. Initialize Graph & remove self-loops
G = nx.from_pandas_adjacency(co_matrix)
G.remove_edges_from(nx.selfloop_edges(G))

# Filter edges to keep meaningful co-occurrences (weight >= 1)
threshold = 1
edges_to_keep = [
    (u, v) for u, v, d in G.edges(data=True) if d['weight'] >= threshold
]
G_sub = G.edge_subgraph(edges_to_keep).copy()
G_sub.remove_nodes_from(list(nx.isolates(G_sub)))

nodes_in_graph = list(G_sub.nodes())
counts_in_graph = np.array([genre_counts[node] for node in nodes_in_graph])

# 4. Scale Node Sizes (Slightly larger range for better visual hierarchy)
min_size, max_size = 900, 4800
c_min, c_max = counts_in_graph.min(), counts_in_graph.max()
normalized = (counts_in_graph - c_min) / (c_max - c_min + 1e-5)
node_sizes = min_size + normalized * (max_size - min_size)

# Scale line thickness for edges
edge_weights = [G_sub[u][v]['weight'] * 0.9 for u, v in G_sub.edges()]

# 5. Plotting with Autumn Styling
plt.figure(figsize=(14, 11), facecolor='#FAF6F0')  # Warm off-white background

# Layout with increased k-spacing to pull dense central nodes apart
pos = nx.spring_layout(G_sub, k=1.05, iterations=60, seed=42)

# Draw Edges: Warm terracotta/copper line color + fixed warning (removed connectionstyle)
nx.draw_networkx_edges(
    G_sub,
    pos,
    width=edge_weights,
    edge_color='#D9825B',  # Soft autumn copper edge
    alpha=0.35,  # Subdued transparency to keep center readable
)

# Draw Nodes: Use 'YlOrBr' (Yellow-Orange-Brown) or 'autumn' colormap
nodes = nx.draw_networkx_nodes(
    G_sub,
    pos,
    nodelist=nodes_in_graph,
    node_size=node_sizes,
    node_color=counts_in_graph,
    cmap=plt.cm.YlOrBr,  # Warm Autumn Gradient (Golden Yellow -> Amber -> Burnt Orange -> Dark Rust)
    alpha=0.92,
    edgecolors='#5C2414',  # Deep rust border outline
    linewidths=1.8,
)

# Draw Labels: Deep espresso/chocolate text, slightly smaller font for clean fitting
nx.draw_networkx_labels(
    G_sub,
    pos,
    font_size=9.5,  # Adjusted font size to fit inside node centers
    font_family='sans-serif',
    font_weight='bold',
    font_color='#2B1704',  # Deep autumn brown font color
)

# Colorbar Styling
cbar = plt.colorbar(nodes, shrink=0.75, pad=0.02)
cbar.set_label('Total Genre Occurrences', fontsize=11, fontweight='bold', color='#4A2511')
cbar.ax.tick_params(labelsize=10, colors='#4A2511')
cbar.outline.set_edgecolor('#5C2414')

plt.title(
    'Netflix K-Drama Genre Co-Occurrence & Relationship Network',
    fontsize=16,
    fontweight='bold',
    pad=25,
    color='#3D1A08',
)
plt.axis('off')
plt.tight_layout()
plt.show()

### Upset Plot

This is a graph that I had not seen before, it was developed in 2014. The Wikipedia page shows an example that was used for showing genre combinations in films. 

In [ ]:
pip install upsetplot

In [ ]:
import warnings
from upsetplot import from_indicators, plot
import matplotlib.pyplot as plt
warnings.simplefilter(action='ignore', category=FutureWarning)


# 1. One-hot encode the list of genres
genres_encoded = netflix['Genres'].explode().str.get_dummies().groupby(level=0).sum().astype(bool)

# 2. Generate and plot UpSet diagram
upset_data = from_indicators(genres_encoded)
plot(upset_data, subset_size='sum', show_counts=True)
plt.title("Genre Combination Frequencies")
plt.show()

**Note:** I showed these different graphs to demostrate that sometimes you need to try out various visualizations to find something that you like or tha fits that story that you are telling with the data.

## 3. Finding NERs in text

If we look at the Kdrama text descriptions, they have a lot of character names. Being able to automatically extract them from text like this is relevant when we want to know about characters that commenters are talking about in Reddit. 

We will look into **Named Entity Recognition** as a way for doing **information extraction**.

We will do this using the Python library [**spaCy**](https://spacy.io/). 

### Installations

You need to install these two things first:

In [ ]:
pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm

**What's in a name?**

```
en  _  core  _  web  _  sm
  │       │        │      │
  │       │        │      └── Size (Small)
  │       │        └──────── Context/Genre (Web text)
  │       └──────────────── Model Type (Core pipeline)
  └──────────────────────── Language (English)
```

**core**: Indicates a general-purpose core NLP pipeline. It means the model includes a full suite of standard components: Part-of-Speech (POS) tagging, dependency parsing, lemmatization, and Named Entity Recognition (NER).

### Working with spaCY

I asked Gemini to create a synthetic document that contains a lot of information that could be extracted.

In [ ]:
text = """Yesterday at 8:30 AM, Dr. Aris Thorne, a leading researcher at Aegis Dynamics in London, met with 
representatives from the United Nations to present Project Titan, a $1.2 million quantum initiative funded partly 
by 25% of the European tech grant. Standing near the Thames River, he highlighted that the project—conceived during 
Spring 2024 and detailed in his acclaimed book, The Neural Horizon—had already reduced processing errors by 40 percent 
over a 50-mile radius. Demonstrating the system on an Apple iPad while playing Beethoven’s 5th Symphony in the background, 
he announced plans to expand operations across Asia before the first phase concludes next Tuesday."""

In [ ]:
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

doc = nlp(text)

doc.ents

Although we are seeing a tuple of strings, each item is an object that has many properties:

In [ ]:
type(doc.ents[0])

In [ ]:
# Code written by Gemini

entities = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]

for txt, label, description in entities:
    print(f"{txt:<20} | {label:<10} | {description}")

### Let's try a bigger model

We first need to do some installations, these might take time. We are downloading a "transfomer" model.

In [ ]:
!python -m spacy download en_core_web_trf

In [ ]:
pip install spacy-curated-transformers

In [ ]:
nlp = spacy.load("en_core_web_trf" )

doc = nlp(text)

entities = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]
for txt, label, description in entities:
    print(f"{txt:<20} | {label:<10} | {description}")

### Compare the two models

I asked Gemini to write code to compare the outputs of the two models:

In [ ]:
nlp_sm = spacy.load("en_core_web_sm")
nlp_trf = spacy.load("en_core_web_trf")

doc_sm = nlp_sm(text)
doc_trf = nlp_trf(text)

# Convert entities to dictionaries mapping (text, label)
ents_sm = {(ent.text, ent.label_) for ent in doc_sm.ents}
ents_trf = {(ent.text, ent.label_) for ent in doc_trf.ents}

print(f"{'TEXT':<22} | {'en_core_web_sm':<15} | {'en_core_web_trf':<15}")
print("-" * 60)

all_spans = sorted(
    list(set([e.text for e in doc_sm.ents] + [e.text for e in doc_trf.ents]))
)

for span in all_spans:
    label_sm = next((lbl for txt, lbl in ents_sm if txt == span), "---")
    label_trf = next((lbl for txt, lbl in ents_trf if txt == span), "---")
    print(f"{span:<22} | {label_sm:<15} | {label_trf:<15}")

**Verdict:** They are almost identicial on most words. The transformer model recognized Apple as an organization, but both struggled with Beethoven's 5th Symphony. It is possible to create explicit rules to deal with such issues.

### Creating entity rules

In [ ]:
import spacy
from spacy.pipeline import EntityRuler

nlp = spacy.load("en_core_web_sm")

# Create a rule-based entity overwriter before/after the NER step
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Add a rule for classical works with possessives or numbers
patterns = [
    {
        "label": "WORK_OF_ART",
        "pattern": [
            {"POS": "PROPN"},
            {"LOWER": "’s"},
            {"LIKE_NUM": True},
            {"LOWER": "symphony"},
        ],
    }
]
ruler.add_patterns(patterns)

doc = nlp("They were playing Beethoven’s 5th Symphony in the background.")
for ent in doc.ents:
    print(ent.text, "|", ent.label_)

Now let's try it with our text:

In [ ]:
nlp = spacy.load("en_core_web_trf" )

ruler = nlp.add_pipe("entity_ruler", before="ner")

# Add a rule for classical works with possessives or numbers
patterns = [
    {
        "label": "WORK_OF_ART",
        "pattern": [
            {"POS": "PROPN"},
            {"LOWER": "’s"},
            {"LIKE_NUM": True},
            {"LOWER": "symphony"},
        ],
    }
]
ruler.add_patterns(patterns)

doc = nlp(text)

entities = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]
for txt, label, description in entities:
    print(f"{txt:<20} | {label:<10} | {description}")

It worked here too.

### Kdrama example

We will use one of the Kdrama description snippets to see what the NER can do for us.

First we'll try the transformer model and then the small model.

In [ ]:
kdrama = """The app Love Alarm is created to notify users when someone within 10 meters likes them, 
without revealing who it is. Kim Jo Jo, a high school student who lost her parents at a young age, 
lives with her aunt's family, which makes her life difficult. She doesn't download the app because her 
phone is too old. Hwang Sun Oh, a popular student from a wealthy but emotionally distant family, attends 
the same school as Kim Jo Jo. His best friend, Lee Hye Yeong, likes Kim Jo Jo, and Hwang Sun Oh kisses 
her to confirm his feelings. (Source: MyDramaList) ~~ Adapted from the webtoon "Johahamyeon Ullineun" 
(좋아하면 울리는) by Chun Kye Young (천계영)."""

In [ ]:
nlp = spacy.load("en_core_web_trf" )

doc = nlp(kdrama)

entities = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]
for txt, label, description in entities:
    print(f"{txt:<20} | {label:<10} | {description}")

The smaller model:

In [ ]:
nlp = spacy.load("en_core_web_sm" )

doc = nlp(kdrama)

entities = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]
for txt, label, description in entities:
    print(f"{txt:<20} | {label:<10} | {description}")

There are differences between the two models. A few noticeable ones:

* the smaller model called "Love Alarm" a person, not a product
* the smaller model didn't catch all names with three parts such as Hwang Sun Oh
* the smaller model didn't catch "MyDramaList" as an organization
* the smaller model created two versions of the same name: Kim Jo Jo. and Kim Jo Jo (no period)
* the transformer model didn't catch 10 meters

### A new kdrama example

In [ ]:
text = """Han Geu Ru is an autistic 20-year-old guy. He works for his father’s business “Move To Heaven,” 
a company that specializes in crime scene cleanup, where they also collect and arrange items left by 
deceased people, and deliver them to the bereaved family. When Geu Ru's father dies, Geu Ru's guardianship 
passes to his uncle, ex-convict Cho Sang Gu, who is a martial arts fighter in underground matches. 
Per the father's will, Sang Gu must care for and work with Geu Ru in “Move To Heaven” for three months 
to gain full guardianship and claim the inheritance. Eyeing money, Sang Gu agrees to the conditions and moves 
in. (Source: MyDramaList) ~~ Adapted from the nonfiction essay "Things Left Behind" by professional trauma 
cleaner Kim Sae Byul."""

nlp = spacy.load("en_core_web_trf" )
doc = nlp(text)

entities = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]
for txt, label, description in entities:
    print(f"{txt:<20} | {label:<10} | {description}")

**More issues**

Here are some other issues to notice here:

* Move to Heaven is labeled differently in the two contexts: first as an organization (correct), then as a work of art (incorrect). 
* The phrase 20-year-old is labeled as DATE, when it is not
* MyDramaList that was labeled as ORG in a prior example, is labeled WORK_OF_ART here

After these examples, we can see that the transformers are pretty good, **but they are not consistent and they are not perfect.**

<a id="sec4"></a>
## 4. Semantic Similarity with `spaCy`

Given that spaCy works with transformers (or other NLP models), let's see what other useful things it can do for us in the context of our project.

### 4.1 Lexical similarity

I asked Gemini to create four sentences for us to compare. You might remember that earlier in the semester I ask you to calculate the cosine similarity for document with and without stop words. Let's remind ourselves of how that works. Code was written by Gemini, but I simplified it for our purposes.

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------
# 1. Setup Data & Cleaning Function
# ---------------------------------------------------------
sentences = [
    "Yoon Seri and Ri Jeonghyeok fall deeply in love amidst secret meetings in North Korea.",  # Romance 1
    "Haesoo and Wang So share a passionate, tragic romance in the royal palace.",  # Romance 2
    "Vincenzo Cassano hunts down corrupt politicians to avenge his family's brutal murder.",  # Thriller 1
    "Detective Cha Jiwon relentlessly pursues a notorious serial killer operating in Seoul.",  # Thriller 2
]

labels = ["Romance 1", "Romance 2", "Thriller 1", "Thriller 2"]


def clean_text(text):
    """Lowercases text and strips punctuation."""
    text = re.sub(r"[^\w\s]", "", text.lower())
    return text.strip()

cleaned_sentences = [clean_text(s) for s in sentences]
cleaned_sentences

Create a helper function so that it's easy to understand the results of the cosine similarity calculation.

In [ ]:
def compute_and_print_matrix(matrix, title):
    """Formats and prints pairwise similarity matrices neatly using Pandas."""
    df = pd.DataFrame(matrix, index=labels, columns=labels)
    print(f"\n--- {title} ---")
    print(df.round(3))

#### Calculate cosine similarity on two conditions: with and without stop words

In [ ]:
# ---------------------------------------------------------
# Step 2: Cosine Similarity with CountVectorizer (All Words)
# ---------------------------------------------------------
vectorizer_all = CountVectorizer()
X_all = vectorizer_all.fit_transform(cleaned_sentences)

cosine_all_mat = cosine_similarity(X_all)
compute_and_print_matrix(
    cosine_all_mat, "1. Cosine Similarity (CountVectorizer - All Words)"
)

# ---------------------------------------------------------
# Step 3: Cosine Similarity with CountVectorizer (Stop Words Removed)
# ---------------------------------------------------------
vectorizer_no_stop = CountVectorizer(stop_words="english")
X_no_stop = vectorizer_no_stop.fit_transform(cleaned_sentences)

cosine_no_stop_mat = cosine_similarity(X_no_stop)
compute_and_print_matrix(
    cosine_no_stop_mat,
    "2. Cosine Similarity (CountVectorizer - Stop Words Removed)",
)

**Note:** Notice how the similarity is entirely based on the stop words. Once we remove them, the similarity goes to 0. Lexical similarity is useless when two pieces of text don't share non-stop words in common.

### CountVectorizer

This is a class that creates the document-term matrix that we created manually in prior assignments. By default, it is stored as a sparse matrix, see below:

In [ ]:
X_all

But, we can write some code to convert it into a pandas to see its content:

In [ ]:
# Extract feature names (vocabulary) from the fit vectorizer
feature_names = vectorizer_all.get_feature_names_out()

# Convert the sparse matrix X_all to a dense array and wrap in a DataFrame
df_X_all = pd.DataFrame(X_all.toarray(), index=labels, columns=feature_names)

df_X_all

### 4.2 Semantic similarity

More advanced NLP models or pipelines can capture **semantic similarity**.

* We'll see **static vector embeddings** (circa 2014 NLP)
* We'll see **contextual embeddings with LLMs** (after 2020 NLP)

The model below is an NLP pipeline that uses GloVe (static word embeddings).


#### 4.2.1 Semantic similarity with static word embeddings

In [ ]:
!python -m spacy download en_core_web_lg

In [ ]:
import spacy

# Load SpaCy model with static vector embeddings
nlp = spacy.load("en_core_web_lg")

sentences = [
    "Yoon Seri and Ri Jeonghyeok fall deeply in love amidst secret meetings in North Korea.",
    "Haesoo and Wang So share a passionate, tragic romance in the royal palace.",
    "Vincenzo Cassano hunts down corrupt politicians to avenge his family's brutal murder.",
    "Detective Cha Jiwon relentlessly pursues a notorious serial killer operating in Seoul.",
]

docs = [nlp(text) for text in sentences]

# Compare two documents directly using SpaCy's native similarity function
similarity_score = docs[0].similarity(docs[1])
print(f"Similarity between Romance 1 and Romance 2: {similarity_score:.4f}")

# Compute 4x4 matrix using native SpaCy similarity
import numpy as np

sim_matrix = np.array([[doc1.similarity(doc2) for doc2 in docs] for doc1 in docs])
compute_and_print_matrix(sim_matrix, "Semantic Similarity with static embeddings")


**Note:** This is a good result, the two romance sentences are the most similar, as we expected. However, Thriller 1 and Thriller 2 are as similar to one another as they are to Romance 2, most likely because Romance 2 contains words like "tragic" and "royal palace", which often are part of thriller / crime. 

#### 4.2 Semantic similarity with transformers (contenxtual word embeddings)

Notice that we are using the transformer model here.

In [ ]:
# Load transformer model
nlp = spacy.load("en_core_web_trf")

# Define 4 sentences (2 Romance, 2 Thriller/Crime)
sentences = [
    "Yoon Seri and Ri Jeonghyeok fall deeply in love amidst secret meetings in North Korea.",  # Romance 1
    "Haesoo and Wang So share a passionate, tragic romance in the royal palace.",  # Romance 2
    "Vincenzo Cassano hunts down corrupt politicians to avenge his family's brutal murder.",  # Thriller 1
    "Detective Cha Jiwon relentlessly pursues a notorious serial killer operating in Seoul.",  # Thriller 2
]

# Process documents
docs = [nlp(text) for text in sentences]

It turns out that the representation with Transformers in **spaCy** doesn't generate vectors by default, so we cannot use the same code as before. Instead, we need to write code to generate the vector represenation from the model. This is why below we are using a helper function to do that.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from util_trf import get_trf_vector

# Extract vectors and compute similarity matrix
doc_vectors = np.array([get_trf_vector(doc) for doc in docs])
sim_matrix = cosine_similarity(doc_vectors)

compute_and_print_matrix(sim_matrix, "Semantic Similarity with contextual embeddings")

**Verdict:** This model is not good for measuring similarity and we should not be using it for that task.

<a id="sec5"></a>
## 5. Sentence Embeddings

We'll see a different model for calculating semantic similarity with sentence embeddings.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Downloads on first run, loads from cache on subsequent runs
model = SentenceTransformer("all-MiniLM-L6-v2")

# Verify model loaded successfully
print("Model loaded. Vector dimension:", model.get_sentence_embedding_dimension())

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Sentences
sentences = [
    "Yoon Seri and Ri Jeonghyeok fall deeply in love amidst secret meetings in North Korea.",  # Romance 1
    "Haesoo and Wang So share a passionate, tragic romance in the royal palace.",  # Romance 2
    "Vincenzo Cassano hunts down corrupt politicians to avenge his family's brutal murder.",  # Thriller 1
    "Detective Cha Jiwon relentlessly pursues a notorious serial killer operating in Seoul.",  # Thriller 2
]

labels = ["Romance 1", "Romance 2", "Thriller 1", "Thriller 2"]

# 3. Generate fine-tuned embeddings & compute similarity matrix
embeddings = model.encode(sentences)
sim_matrix = cosine_similarity(embeddings)

# 4. Display as DataFrame
df_sim = pd.DataFrame(sim_matrix, index=labels, columns=labels)
df_sim.round(3)

**Note:** Notice that the values here are much smaller than in the prior models. Romance1 and Romance2 are the most similar, but they are very close to Romance2 and Thriller2. 